# W2-02 (EEC-24): transcribe the 1,500 new training clips
Fresh Colab **T4 GPU** runtime. Self-contained: it downloads the public repo and the public
ADI-20 audio itself, and saves everything to a folder in **your own** Google Drive
(`MyDrive/W2-02-text-scaling`). No GitHub push or shared Drive needed.

Same Jais revision, prompt, batch size (2), token cap (128) and float16 as W1-13. Only the
1,500 new clips are transcribed. Nothing here trains or tunes anything.

**If Colab disconnects:** rerun cells 1–3, skip 4 if audio finished, then rerun 6.
Finished clips are cache hits and are not regenerated. Stay on a T4: the cache key
includes GPU and package versions, so switching GPU type would redo everything.

In [ ]:
# 1. Drive + GPU (outputs go to your own Drive so they survive disconnects)
from google.colab import drive
from pathlib import Path
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
DATA_ROOT = Path('/content/drive/MyDrive/W2-02-text-scaling')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)
print('Saving everything under:', DATA_ROOT)

In [ ]:
# 2. Pinned source checkout, logging helpers, dependencies
import subprocess, sys, tempfile, json, shutil, time
from datetime import datetime, timezone

# main after Week 1 (PR #17). Jais adapter/exporter/evaluator are byte-identical to
# the W1-13 pinned commit 26b36593..., so transcription behaviour is unchanged.
PINNED_SOURCE_COMMIT = 'e66ae0e255a9c7f90ce7c2d8b996344b8e348afd'
REPOSITORY_URL = 'https://github.com/alizbeeb5/Arabic-Dialect-Identification.git'
RUN_ROOT = DATA_ROOT / 'runs/w2_02_text_scaling' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOG_ROOT = RUN_ROOT / 'logs'
LOG_ROOT.mkdir(parents=True, exist_ok=False)

def run_logged(argv, *, cwd=None, allowed=(0,), log_name='command'):
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    log_path = LOG_ROOT / f'{stamp}-{log_name}.log'
    print(f'Running {log_name}; log: {log_path}', flush=True)
    with log_path.open('x', encoding='utf-8') as log, subprocess.Popen(
            list(map(str, argv)), cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            code = process.wait()
        except BaseException:
            if process.poll() is None:  # don't leave inference running after an interrupt
                process.terminate()
                try: process.wait(timeout=5)
                except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if code not in allowed:
        raise RuntimeError(f'{log_name} exited {code}; see {log_path}')
    return code

def command(script, args, allowed=(0,)):
    return run_logged([sys.executable, '-u', REPO_ROOT / script, *args],
                      cwd=REPO_ROOT, allowed=allowed, log_name=Path(script).stem)

REPO_ROOT = Path(tempfile.mkdtemp(prefix='w2-02-'))
for step, argv in [('git-init', ['git', 'init', '--quiet', REPO_ROOT]),
                   ('git-remote', ['git', '-C', REPO_ROOT, 'remote', 'add', 'origin', REPOSITORY_URL]),
                   ('git-fetch', ['git', '-C', REPO_ROOT, 'fetch', '--depth', '1', 'origin', PINNED_SOURCE_COMMIT]),
                   ('git-checkout', ['git', '-C', REPO_ROOT, 'checkout', '--quiet', '--detach', 'FETCH_HEAD'])]:
    run_logged(argv, log_name=step)
SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO_ROOT), 'rev-parse', 'HEAD'], text=True).strip()
assert SOURCE_COMMIT == PINNED_SOURCE_COMMIT, SOURCE_COMMIT

for req in ['environments/jais.txt', 'requirements-data.txt']:
    run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_ROOT / req], log_name='pip')

AUDIO_ROOT = DATA_ROOT / 'audio/adi20_micro/c8da454c9f474bb075d3fe608940ee29e8ccfa43'
JAIS_CACHE = DATA_ROOT / 'cache/jais_adapter_module_v1'   # same layout as W1-13
HF_CACHE = Path('/content/hf-jais-w2-02')
print('Source:', SOURCE_COMMIT); print('Outputs:', RUN_ROOT)

In [ ]:
# 3. Write the selection script, rebuild the nested 2,000 / new 1,500 manifests,
#    and check they match the agreed files byte for byte
EXPAND_SCRIPT = r'''#!/usr/bin/env python3
"""W2-02 (EEC-24): build the nested 2,000-clip training manifest for text scaling.

Reruns the frozen W1-07 selection (src/data/freeze_splits.py, same seed, same
eligibility rules, same audited full-release manifests) with 100 clips per label
instead of 25. The round-robin draw order does not depend on the quota, so the
first 25 picks per label are exactly the original pilot: the 2,000 IDs strictly
contain the original 500 and the 200 dev IDs are unchanged. Both properties are
checked here and the script fails if either does not hold. No dev performance is
read at any point.

Writes (under --output-root, default repository root):
  data/manifests/train_scaling_2000.jsonl (+ .metadata.json)  all 2,000 rows
  data/manifests/train_scaling_new_1500.jsonl                 only the new rows to transcribe
The frozen W1-07 files (train_pilot.jsonl, dev_pilot.jsonl, split_freeze.json) are not touched.

Reproduce:
  python src/data/expand_train_pilot.py --manifest-dir src/data/manifests
"""

import argparse
import json
import sys
import tempfile
from collections import Counter
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))
import freeze_splits as fs  # noqa: E402

TARGET_PER_LABEL = 100
EXPANSION_VERSION = "w2-02-text-scaling-v1"


def parse_args(argv=None):
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--manifest-dir", type=Path, default=fs.REPO_ROOT / "src/data/manifests",
                        help="Audited full-release train/dev manifests used by W1-07")
    parser.add_argument("--frozen", type=Path,
                        default=fs.REPO_ROOT / "data/manifests/split_freeze.json",
                        help="The W1-07 split_freeze.json to extend")
    parser.add_argument("--labels", type=Path, default=fs.REPO_ROOT / "configs/labels.json")
    parser.add_argument("--output-root", type=Path, default=fs.REPO_ROOT)
    return parser.parse_args(argv)


def main(argv=None):
    args = parse_args(argv)
    frozen = json.loads(args.frozen.read_text(encoding="utf-8"))
    labels = fs.load_labels(args.labels)

    with tempfile.TemporaryDirectory() as tmp:
        fs.run(fs.parse_args([
            "--manifest-dir", str(args.manifest_dir), "--labels", str(args.labels),
            "--seed", str(frozen["seed"]), "--dev-total", str(frozen["dev_total"]),
            "--train-per-label", str(TARGET_PER_LABEL), "--output-root", tmp]))
        out = Path(tmp) / "data/manifests"
        expanded = json.loads((out / "split_freeze.json").read_text(encoding="utf-8"))
        rows = [json.loads(line) for line in
                (out / "train_pilot.jsonl").read_text(encoding="utf-8").splitlines() if line]

    # Guards: same source, nested training IDs, identical dev IDs, balanced, train-only.
    if expanded["source_manifest_sha256"] != frozen["source_manifest_sha256"]:
        raise SystemExit("Source manifests differ from the ones W1-07 was frozen from")
    if expanded["labels_sha256"] != frozen["labels_sha256"]:
        raise SystemExit("labels.json differs from the W1-07 freeze")
    original = set(frozen["train_pilot_ids"])
    ids = [r["clip_id"] for r in rows]
    if len(ids) != len(set(ids)) or not original <= set(ids):
        raise SystemExit("Expanded set does not strictly contain the original 500 training IDs")
    if expanded["dev_pilot_ids"] != frozen["dev_pilot_ids"]:
        raise SystemExit("Dev pilot IDs changed")
    if set(ids) & set(frozen["dev_pilot_ids"]) or any(r["split"] != "train" for r in rows):
        raise SystemExit("Expanded set contains non-training clips")
    counts = Counter(r["label"] for r in rows)
    if any(counts[label] != TARGET_PER_LABEL for label in labels):
        raise SystemExit(f"Unbalanced expansion: {dict(counts)}")

    new_rows = [r for r in rows if r["clip_id"] not in original]
    full_bytes, new_bytes = fs.jsonl_bytes(rows), fs.jsonl_bytes(new_rows)
    manifest_out = args.output_root / "data/manifests"
    manifest_out.mkdir(parents=True, exist_ok=True)
    (manifest_out / "train_scaling_2000.jsonl").write_bytes(full_bytes)
    (manifest_out / "train_scaling_new_1500.jsonl").write_bytes(new_bytes)

    # Sidecars follow the train_pilot.metadata.json schema so the existing Jais
    # adapter/exporter, evaluator and TF-IDF scripts accept them unchanged.
    base = json.loads((args.frozen.parent / "train_pilot.metadata.json").read_text(encoding="utf-8"))
    identity = {"expansion_version": EXPANSION_VERSION, "nested_in": frozen["split_version"],
                "seed": frozen["seed"], "train_per_label": TARGET_PER_LABEL,
                "labels_sha256": frozen["labels_sha256"],
                "source_manifest_sha256": frozen["source_manifest_sha256"],
                "train_scaling_2000_sha256": fs.sha256_bytes(full_bytes),
                "train_scaling_new_1500_sha256": fs.sha256_bytes(new_bytes)}
    split_hash = fs.sha256_bytes(fs.canonical(identity))
    split_version = f"{EXPANSION_VERSION}-{split_hash[:12]}"
    provenance = {
        "issue": "EEC-24 / W2-02",
        "selection": {"method": "freeze_splits.sample_balanced rerun with train_per_label=100",
                      "seed": frozen["seed"], "freeze_version": fs.FREEZE_VERSION,
                      "freeze_code_sha256": fs.sha256_file(fs.__file__),
                      "expand_code_sha256": fs.sha256_file(__file__),
                      "dev_performance_used": False},
        "nested_in": {"split_version": frozen["split_version"],
                      "train_pilot_sha256": frozen["train_pilot_sha256"],
                      "contains_all_original_ids": True},
        "dev_pilot_unchanged": {"dev_pilot_sha256": frozen["dev_pilot_sha256"], "verified": True},
        "expansion_identity": identity,
    }

    def sidecar(payload, chosen, part):
        return {**base, "split_version": split_version, "split_sha256": split_hash,
                "manifest_sha256": fs.sha256_bytes(payload), "clips": len(chosen),
                "label_counts": {label: sum(r["label"] == label for r in chosen) for label in labels},
                "evaluation_label_sets": {"train": sorted({r["label"] for r in chosen}, key=labels.index)},
                "scaling_part": part, "w2_02_provenance": provenance}

    for name, payload, chosen, part in (
            ("train_scaling_2000", full_bytes, rows, "all_2000_nested"),
            ("train_scaling_new_1500", new_bytes, new_rows, "new_1500_only")):
        (manifest_out / f"{name}.metadata.json").write_bytes(fs.pretty(sidecar(payload, chosen, part)))
    print(f"Wrote {len(rows)} training clips ({len(new_rows)} new, 75 per label) to {manifest_out}")
    print(f"split_version {split_version}")
    print(f"train_scaling_2000.jsonl     sha256 {fs.sha256_bytes(full_bytes)}")
    print(f"train_scaling_new_1500.jsonl sha256 {fs.sha256_bytes(new_bytes)}")


if __name__ == "__main__":
    main()
'''
(REPO_ROOT / 'src/data/expand_train_pilot.py').write_text(EXPAND_SCRIPT, encoding='utf-8')
command('src/data/expand_train_pilot.py', ['--manifest-dir', REPO_ROOT / 'src/data/manifests'])

import hashlib
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
M = REPO_ROOT / 'data/manifests'
NEW_MANIFEST, NEW_META = M / 'train_scaling_new_1500.jsonl', M / 'train_scaling_new_1500.metadata.json'
assert sha(M / 'train_scaling_2000.jsonl') == '41514213e14bfbd82d4fbf8cfe6eae04a2faf4f878509a15758f431dc41f7929'
assert sha(NEW_MANIFEST) == 'cf727f130e46be2e091386b5a5ed55336de5d5c61ae71fb4df600bae4f82ce94'
NEW_ROWS = [json.loads(l) for l in NEW_MANIFEST.read_text().splitlines() if l.strip()]
print('Manifests match.', len(NEW_ROWS), 'new clips to transcribe.')
(RUN_ROOT / 'manifests').mkdir()
for f in M.glob('train_scaling_*'): shutil.copy(f, RUN_ROOT / 'manifests' / f.name)
shutil.copy(REPO_ROOT / 'src/data/expand_train_pilot.py', RUN_ROOT / 'manifests')

In [ ]:
# 4. Download only the 1,500 new WAVs (verified by SHA256; resumable, skips files already present)
t0 = time.perf_counter()
command('src/data/fetch_pilot_audio.py', ['--audio-root', AUDIO_ROOT, '--manifests', NEW_MANIFEST])
FETCH_SECONDS = time.perf_counter() - t0
missing = [r['clip_id'] for r in NEW_ROWS if not (AUDIO_ROOT / r['audio_path']).is_file()]
assert not missing, f'{len(missing)} WAVs still missing'
print(f'All {len(NEW_ROWS)} WAVs present. Fetch took {FETCH_SECONDS/60:.1f} min')

In [ ]:
# 5. 20-clip smoke (first new clip per dialect) to check the environment before the long run
def run_split(name, manifest, metadata):
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    out = RUN_ROOT / f'{name}-{attempt}'
    status = command('src/models/jais_adapter.py', [
        '--manifest', manifest, '--metadata', metadata, '--audio-root', AUDIO_ROOT,
        '--cache-root', JAIS_CACHE, '--model-cache', HF_CACHE, '--output-dir', out,
        '--dtype', 'float16', '--batch-size', '2', '--max-new-tokens', '128'], allowed=(0, 2))
    exported = RUN_ROOT / f'{name}-export-{attempt}'
    command('src/experiments/jais_pilot.py', ['export', '--manifest', manifest,
        '--metadata', metadata, '--split', 'train', '--run-dir', out, '--output-dir', exported])
    summary = json.loads((exported / 'summary.json').read_text())
    res = summary['resource_record']
    print('Adapter exit:', status)
    print('Transcript status counts:', summary['transcript_counts'])
    print('Counts (cache hits / new outputs):', res['counts'])
    print('Refused / truncated / unknown:', summary['refused'], summary['truncated'], summary['truncation_unknown'])
    print(f"Wall time: {res['wall_seconds']/60:.1f} min | peak GPU: {res['peak_gpu_allocated_gib']} GiB")
    print('Export:', exported)
    return out, exported

SMOKE_INPUT = RUN_ROOT / 'smoke-input'
command('src/experiments/jais_pilot.py', ['prepare-smoke', '--manifest', NEW_MANIFEST,
        '--metadata', NEW_META, '--output-dir', SMOKE_INPUT])
SMOKE_RUN, SMOKE_EXPORT = run_split('smoke', SMOKE_INPUT / 'smoke.jsonl', SMOKE_INPUT / 'smoke.metadata.json')

## Check the smoke output before continuing
Expect 20 rows, mostly `ok` transcripts, few or no refusals. Paste the printed lines
back into the chat. Settings must NOT change between smoke and the full run.

In [ ]:
# 6. Full run on the 1,500 new clips (~51-61 min on T4 by the W1 estimate; the 20 smoke clips are cache hits)
NEW_RUN, NEW_EXPORT = run_split('train-new1500', NEW_MANIFEST, NEW_META)
print('Keep this path for the TF-IDF step:', NEW_EXPORT)

# Part 2: original 500 train + 200 dev transcripts
The W1-13 export archive (`task13_evidence.zip`) was not reachable from this run, so the
original 500 training and 200 dev clips were re-transcribed here with the identical
adapter, revision and settings. The resulting 500-clip text baseline reproduces the Week 1
result (50/200, identical macro-F1); see the report for the byte-level check.

In [ ]:
# B1. Same Jais environment as the 1,500 run, then fetch the 700 original pilot WAVs
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_ROOT / 'environments/jais.txt'], log_name='pip-jais-restore')
M = REPO_ROOT / 'data/manifests'
t0 = time.perf_counter()
command('src/data/fetch_pilot_audio.py', ['--audio-root', AUDIO_ROOT,
        '--manifests', M / 'train_pilot.jsonl', M / 'dev_pilot.jsonl'])
print(f'Fetch took {(time.perf_counter() - t0)/60:.1f} min')

In [ ]:
# B2. Re-transcribe original 500 train + 200 dev with the identical adapter settings
def run_split_any(name, manifest, metadata, split):
    attempt = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    out = RUN_ROOT / f'{name}-{attempt}'
    status = command('src/models/jais_adapter.py', [
        '--manifest', manifest, '--metadata', metadata, '--audio-root', AUDIO_ROOT,
        '--cache-root', JAIS_CACHE, '--model-cache', HF_CACHE, '--output-dir', out,
        '--dtype', 'float16', '--batch-size', '2', '--max-new-tokens', '128'], allowed=(0, 2))
    exported = RUN_ROOT / f'{name}-export-{attempt}'
    command('src/experiments/jais_pilot.py', ['export', '--manifest', manifest,
        '--metadata', metadata, '--split', split, '--run-dir', out, '--output-dir', exported])
    summary = json.loads((exported / 'summary.json').read_text())
    res = summary['resource_record']
    print(f'== {name}: exit {status} | transcripts {summary["transcript_counts"]} | '
          f'refused/truncated/unknown {summary["refused"]} {summary["truncated"]} {summary["truncation_unknown"]} | '
          f'{res["wall_seconds"]/60:.1f} min | counts {res["counts"]}')
    return exported

OLD_TRAIN_EXPORT = run_split_any('train-orig500', M / 'train_pilot.jsonl', M / 'train_pilot.metadata.json', 'train')
DEV_EXPORT = run_split_any('dev-orig200', M / 'dev_pilot.jsonl', M / 'dev_pilot.metadata.json', 'dev')

In [ ]:
# B3. Text-model environment and paths
import hashlib
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
run_logged([sys.executable, '-m', 'pip', 'install', '-q', '-r', REPO_ROOT / 'environments/text-pilot.txt'], log_name='pip-text')
NEW_EXPORT = sorted(DATA_ROOT.glob('runs/w2_02_text_scaling/*/train-new1500-export-*'))[-1]
print('Re-transcribed 500 export:', OLD_TRAIN_EXPORT)
print('Re-transcribed dev export:', DEV_EXPORT)
print('New 1,500 export:', NEW_EXPORT)

# Part 3: 500 vs 2,000 text models (CPU)

In [ ]:
%%writefile /content/merge_text_exports.py
#!/usr/bin/env python3
"""W2-02 (EEC-24): combine the W1-13 500-clip and W2-02 1,500-clip transcript exports.

Both inputs are validated with the same loader the text model uses
(src/models/transcript_tfidf.load_export), against their own manifests. The
output is one export for train_scaling_2000 in manifest order, with records
copied unchanged. No transcription, filtering or relabelling happens here.

Usage:
  python src/data/merge_text_exports.py \
    --original-export TRAIN_EXPORT_500 --new-export NEW_EXPORT_1500 --output-dir OUT
"""
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))
from src.models.transcript_tfidf import load_export  # noqa: E402
from src.experiments.jais_pilot import write_jsonl  # noqa: E402
from src.models.jais_adapter import atomic_json, read_json, sha256_file  # noqa: E402

M = ROOT / "data/manifests"
LABELS = ROOT / "configs/labels.json"
PARTS = {
    "original_500": (M / "train_pilot.jsonl", M / "train_pilot.metadata.json"),
    "new_1500": (M / "train_scaling_new_1500.jsonl", M / "train_scaling_new_1500.metadata.json"),
}
TARGET = (M / "train_scaling_2000.jsonl", M / "train_scaling_2000.metadata.json")


def read_rows(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--original-export", type=Path, required=True)
    parser.add_argument("--new-export", type=Path, required=True)
    parser.add_argument("--output-dir", type=Path, required=True)
    args = parser.parse_args(argv)
    if args.output_dir.exists():
        parser.error("Use a new output directory")

    records, targets, sources = {}, {}, {}
    for part, export in (("original_500", args.original_export), ("new_1500", args.new_export)):
        manifest, metadata = PARTS[part]
        part_records, part_targets, _, identity = load_export(export, manifest, metadata, LABELS, "train")
        overlap = set(records) & {r["clip_id"] for r in part_records}
        if overlap:
            raise SystemExit(f"{len(overlap)} clip IDs appear in both exports")
        for record in part_records:
            records[record["clip_id"]] = record
        targets.update(part_targets)
        summary = read_json(Path(export) / "summary.json")
        sources[part] = {
            "export_dir": str(export), "identity": identity, "expected": summary["expected"],
            "transcript_counts": summary["transcript_counts"], "refused": summary["refused"],
            "truncated": summary["truncated"], "truncation_unknown": summary["truncation_unknown"],
            "resource_record": summary.get("resource_record"),
        }

    manifest, metadata = TARGET
    rows = read_rows(manifest)
    if {r["clip_id"] for r in rows} != set(records):
        raise SystemExit("Merged IDs do not equal the train_scaling_2000 manifest")
    meta = read_json(metadata)
    ordered = [records[r["clip_id"]] for r in rows]
    for r in rows:
        if targets[r["clip_id"]] != r["label"]:
            raise SystemExit("Target disagrees with the 2,000 manifest")

    out = args.output_dir
    out.mkdir(parents=True)
    write_jsonl(out / "transcripts.jsonl", ordered)
    write_jsonl(out / "targets.jsonl", [{"clip_id": r["clip_id"], "label": r["label"]} for r in rows])
    summary = {
        "schema_version": "1.0", "split": "train", "expected": len(rows),
        "manifest_sha256": sha256_file(manifest), "metadata_sha256": sha256_file(metadata),
        "labels_sha256": sha256_file(LABELS), "split_version": meta["split_version"],
        "dataset_version": meta["dataset_version"],
        "transcript_counts": dict(Counter(r["transcript_status"] for r in ordered)),
        "refused": sum(r["refused"] is True for r in ordered),
        "truncated": sum(r["truncated"] is True for r in ordered),
        "truncation_unknown": sum(r["truncated"] is None for r in ordered),
        "kind": "merged_export_no_new_inference",
        "merge_code_sha256": sha256_file(__file__),
        "sources": sources,
    }
    for name in ("transcripts.jsonl", "targets.jsonl"):
        summary[name + "_sha256"] = sha256_file(out / name)
    atomic_json(out / "summary.json", summary)
    print(json.dumps({"output_dir": str(out), "expected": len(rows),
                      "transcript_counts": summary["transcript_counts"],
                      "by_source": {k: v["transcript_counts"] for k, v in sources.items()}}, indent=2))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/text_scaling_compare.py
#!/usr/bin/env python3
"""W2-02 (EEC-24): compare the 500- and 2,000-clip text models on the frozen 200 dev IDs.

Reads only saved run directories written by src/models/transcript_tfidf.py.
Re-scores both dev prediction files with the shared evaluator, then reports
accuracy, macro-F1, per-dialect recall/F1, training fit, paired correctness
(with an exact two-sided McNemar test) and prediction sources.

Usage:
  python src/analysis/text_scaling_compare.py \
    --train-500 RUN --dev-500 RUN --train-2000 RUN --dev-2000 RUN --output OUT.json
"""
from __future__ import annotations

import argparse
import json
import sys
from collections import Counter
from math import comb
from pathlib import Path

ROOT = Path(__file__).resolve().parents[2]
sys.path.insert(0, str(ROOT))
from src.evaluation import evaluate as ev  # noqa: E402

DEV = ROOT / "data/manifests/dev_pilot.jsonl"
BASELINE = {"correct": 50, "macro_f1": 0.23642323859715164}


def load(run):
    run = Path(run)
    return (json.loads((run / "metrics.json").read_text()),
            json.loads((run / "run_record.json").read_text()),
            {p["clip_id"]: p for p in ev.read_jsonl(run / "predictions.jsonl")})


def mcnemar_exact(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    return min(1.0, 2 * sum(comb(n, i) for i in range(k + 1)) / 2 ** n)


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    for name in ("train-500", "dev-500", "train-2000", "dev-2000"):
        parser.add_argument(f"--{name}", type=Path, required=True)
    parser.add_argument("--output", type=Path, required=True)
    args = parser.parse_args(argv)

    labels = ev.load_labels(ROOT / "configs/labels.json")
    references = ev.validate_references(ev.read_jsonl(DEV), labels, "dev")
    result = {"dev_ids": len(references), "arms": {}}
    preds = {}
    for arm in ("500", "2000"):
        train_m, train_r, _ = load(getattr(args, f"train_{arm}"))
        dev_m, dev_r, dev_p = load(getattr(args, f"dev_{arm}"))
        if set(dev_p) != set(references):
            raise SystemExit(f"{arm}: dev predictions do not cover exactly the 200 frozen IDs")
        rescored = ev.compute_metrics(ev.join_predictions(
            references, ev.validate_prediction_fields(list(dev_p.values()), labels),
            allow_incomplete=False), labels)
        if rescored["correct"] != dev_m["correct"]:
            raise SystemExit(f"{arm}: saved dev metrics disagree with re-scoring")
        if dev_r["model_revision"] != train_r["model_revision"]:
            raise SystemExit(f"{arm}: dev run used a different checkpoint than its training run")
        diag = json.loads((Path(getattr(args, f"train_{arm}")) / "training_diagnostics.json").read_text())
        preds[arm] = dev_p
        result["arms"][arm] = {
            "checkpoint": train_r["model_revision"],
            "training_expected": diag["expected_training_clips"],
            "training_retained": diag["retained_training_clips"],
            "training_excluded": len(diag["excluded_training_ids"]),
            "retained_per_class_min_max": [min(diag["retained_training_class_counts"].values()),
                                           max(diag["retained_training_class_counts"].values())],
            "feature_counts": diag["feature_counts"],
            "fit_seconds": diag["fit_seconds"],
            "train_accuracy_all_ids": train_m["accuracy"],
            "dev_correct": rescored["correct"], "dev_accuracy": rescored["accuracy"],
            "dev_macro_f1": rescored["macro_f1"],
            "dev_prediction_sources": dev_r["prediction_sources"],
            "per_dialect": {c: {"recall": rescored["per_dialect"][c]["recall"],
                                "f1": rescored["per_dialect"][c]["f1"]} for c in labels},
            "dev_predictions_sha256": dev_m["predictions_sha256"],
        }

    base = result["arms"]["500"]
    result["baseline_500_matches_week1"] = (base["dev_correct"] == BASELINE["correct"]
                                            and abs(base["dev_macro_f1"] - BASELINE["macro_f1"]) < 1e-12)
    pairs = Counter()
    flips = []
    for clip_id, ref in references.items():
        a = preds["500"][clip_id]["predicted_label"] == ref["label"]
        b = preds["2000"][clip_id]["predicted_label"] == ref["label"]
        pairs[("both_correct", "only_500_correct", "only_2000_correct", "both_wrong")
              [0 if a and b else 1 if a else 2 if b else 3]] += 1
        if a != b:
            flips.append({"clip_id": clip_id, "label": ref["label"],
                          "pred_500": preds["500"][clip_id]["predicted_label"],
                          "pred_2000": preds["2000"][clip_id]["predicted_label"]})
    result["paired"] = dict(pairs)
    result["mcnemar_exact_p"] = mcnemar_exact(pairs["only_500_correct"], pairs["only_2000_correct"])
    result["flips"] = flips
    result["delta"] = {
        "dev_correct": result["arms"]["2000"]["dev_correct"] - base["dev_correct"],
        "dev_accuracy_points": 100 * (result["arms"]["2000"]["dev_accuracy"] - base["dev_accuracy"]),
        "dev_macro_f1": result["arms"]["2000"]["dev_macro_f1"] - base["dev_macro_f1"],
    }
    result["per_dialect_recall_change"] = {
        c: result["arms"]["2000"]["per_dialect"][c]["recall"] - base["per_dialect"][c]["recall"]
        for c in labels}
    args.output.parent.mkdir(parents=True, exist_ok=True)
    args.output.write_text(json.dumps(result, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")

    print(f"{'':>6} {'retained':>9} {'train acc':>9} {'dev correct':>12} {'dev acc':>8} {'macro-F1':>9}")
    for arm, r in result["arms"].items():
        print(f"{arm:>6} {r['training_retained']:>9} {r['train_accuracy_all_ids']:>9.3f} "
              f"{r['dev_correct']:>8}/200 {r['dev_accuracy']:>8.3f} {r['dev_macro_f1']:>9.4f}")
    print("Baseline reproduces Week 1 (50/200, same macro-F1):", result["baseline_500_matches_week1"])
    print("Paired:", result["paired"], f"| exact McNemar p = {result['mcnemar_exact_p']:.4f}")
    print("Recall change per dialect (2000 - 500):",
          {c: round(v, 1) for c, v in result["per_dialect_recall_change"].items()})
    print("Saved:", args.output)


if __name__ == "__main__":
    main()

In [ ]:
# C3. Install both scripts in the checkout and build the 2,000-clip export (no new inference)
shutil.copy('/content/merge_text_exports.py', REPO_ROOT / 'src/data/merge_text_exports.py')
shutil.copy('/content/text_scaling_compare.py', REPO_ROOT / 'src/analysis/text_scaling_compare.py')
OUT = RUN_ROOT / 'text_scaling'
OUT.mkdir(exist_ok=True)
MERGED_EXPORT = OUT / 'train2000_export'
command('src/data/merge_text_exports.py', ['--original-export', OLD_TRAIN_EXPORT,
        '--new-export', NEW_EXPORT, '--output-dir', MERGED_EXPORT])

In [ ]:
# C4. Same TF-IDF + logistic regression config for both arms; only the training data differs
M = REPO_ROOT / 'data/manifests'
ARMS = {'500': (OLD_TRAIN_EXPORT, M / 'train_pilot.jsonl', M / 'train_pilot.metadata.json'),
        '2000': (MERGED_EXPORT, M / 'train_scaling_2000.jsonl', M / 'train_scaling_2000.metadata.json')}
for arm, (export, manifest, meta) in ARMS.items():
    command('src/models/transcript_tfidf.py', ['train', '--export-dir', export, '--manifest', manifest,
            '--metadata', meta, '--split', 'train', '--config', REPO_ROOT / 'configs/transcript_tfidf.json',
            '--output-dir', OUT / f'train_{arm}'])
    command('src/models/transcript_tfidf.py', ['predict', '--model', OUT / f'train_{arm}/model.joblib',
            '--export-dir', DEV_EXPORT, '--manifest', M / 'dev_pilot.jsonl',
            '--metadata', M / 'dev_pilot.metadata.json', '--split', 'dev', '--output-dir', OUT / f'dev_{arm}'])
print('Done. Outputs in', OUT)

In [ ]:
# C5. Compare on the same 200 dev clips; results saved to Drive
command('src/analysis/text_scaling_compare.py', [
    '--train-500', OUT / 'train_500', '--dev-500', OUT / 'dev_500',
    '--train-2000', OUT / 'train_2000', '--dev-2000', OUT / 'dev_2000',
    '--output', OUT / 'comparison.json'])
print('All outputs:', OUT)

In [ ]:
# D0. Is the re-run 500 baseline byte-identical to the Week 1 release?
print('500 checkpoint identical to Week 1:', sha(OUT / 'train_500/model.joblib') ==
      '2c82c5c32574d6b58905042d96b59725389aa93bfc3733bb50f3c93471d866f4')
print('500 dev predictions identical to Week 1:', sha(OUT / 'dev_500/predictions.jsonl') ==
      '977e367b92b5235af17ede75ae170c8cb166ec205588f8ec8410f6b97cd6f08e')

In [ ]:
# D1. Bundle the W2-02 evidence (everything except audio, model caches and raw transcripts)
import zipfile
from google.colab import files

BUNDLE = DATA_ROOT / 'w2_02_evidence.zip'
run_dirs = sorted(p for p in RUN_ROOT.iterdir() if p.is_dir() and '-export-' not in p.name
                  and p.name not in ('logs', 'manifests', 'text_scaling', 'smoke-input'))
exports = sorted(RUN_ROOT.glob('*-export-*'))
picked = []
picked += sorted((RUN_ROOT / 'manifests').iterdir())
for d in run_dirs:
    picked += [f for f in (d / 'run_record.json', d / 'predictions.jsonl') if f.exists()]
for d in exports:
    picked += [f for f in (d / 'summary.json', d / 'diagnostics.jsonl') if f.exists()]
TS = RUN_ROOT / 'text_scaling'
for arm in ('train_500', 'train_2000', 'dev_500', 'dev_2000'):
    picked += sorted(f for f in (TS / arm).iterdir() if f.is_file())
picked += [TS / 'train2000_export/summary.json', TS / 'comparison.json']
picked += sorted((RUN_ROOT / 'logs').iterdir())
with zipfile.ZipFile(BUNDLE, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in picked:
        z.write(f, f.relative_to(RUN_ROOT))
print(f'{len(picked)} files, {BUNDLE.stat().st_size/1e6:.1f} MB ->', BUNDLE)
files.download(str(BUNDLE))